## Cleaning the project ideas

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt 

In [5]:
df = pd.read_csv('healthcare_dataset.csv')

print(df.head(6).to_string())

            Name  Age  Gender Blood Type Medical Condition Date of Admission            Doctor                    Hospital Insurance Provider  Billing Amount  Room Number Admission Type Discharge Date   Medication  Test Results
0  Bobby JacksOn   30    Male         B-            Cancer        2024-01-31     Matthew Smith             Sons and Miller         Blue Cross    18856.281306          328         Urgent     2024-02-02  Paracetamol        Normal
1   LesLie TErRy   62    Male         A+           Obesity        2019-08-20   Samantha Davies                     Kim Inc           Medicare    33643.327287          265      Emergency     2019-08-26    Ibuprofen  Inconclusive
2    DaNnY sMitH   76  Female         A-           Obesity        2022-09-22  Tiffany Mitchell                    Cook PLC              Aetna    27955.096079          205      Emergency     2022-10-07      Aspirin        Normal
3   andrEw waTtS   28  Female         O+          Diabetes        2020-11-18       Kevin

In [3]:
## Listing all the column names.
print(df.columns)
print('*'* 40)
print(df.dtypes)
print('*'* 40)
print(df.shape)


Index(['Name', 'Age', 'Gender', 'Blood Type', 'Medical Condition',
       'Date of Admission', 'Doctor', 'Hospital', 'Insurance Provider',
       'Billing Amount', 'Room Number', 'Admission Type', 'Discharge Date',
       'Medication', 'Test Results'],
      dtype='str')
****************************************
Name                      str
Age                     int64
Gender                    str
Blood Type                str
Medical Condition         str
Date of Admission         str
Doctor                    str
Hospital                  str
Insurance Provider        str
Billing Amount        float64
Room Number             int64
Admission Type            str
Discharge Date            str
Medication                str
Test Results              str
dtype: object
****************************************
(55500, 15)


In [6]:
## Data quality audit.
print(df.isna().sum())

print('==' * 50)
print(df.duplicated().sum())

Name                  0
Age                   0
Gender                0
Blood Type            0
Medical Condition     0
Date of Admission     0
Doctor                0
Hospital              0
Insurance Provider    0
Billing Amount        0
Room Number           0
Admission Type        0
Discharge Date        0
Medication            0
Test Results          0
dtype: int64
534


In [9]:
df.apply(lambda x: x.duplicated().sum())

Name                   5508
Age                   55423
Gender                55498
Blood Type            55492
Medical Condition     55494
Date of Admission     53673
Doctor                15159
Hospital              15624
Insurance Provider    55495
Billing Amount         5500
Room Number           55100
Admission Type        55497
Discharge Date        53644
Medication            55495
Test Results          55497
dtype: int64

In [11]:
def data_quality_check(df):

    print("*" * 60)
    print("DATA QUALITY CHECK")
    print("*" * 60)

    ## For the negative age.
    invalid_age = (df["Age"] < 0) | (df["Age"] > 120)

    print("\n1. INVALID AGE")
    print("Count:", invalid_age.sum())

    if invalid_age.sum() > 0:
        print(df.loc[invalid_age, ["Name","Age"]])

    ## Impossible Dates.
    admission_date = pd.to_datetime(
        df["Date of Admission"], errors='coerce'
    )

    discharge_date = pd.to_datetime(
        df["Discharge Date"], errors='coerce'
    )

    invalid_date_format = (
        admission_date.isna() |
        discharge_date.isna()
    )

    invalid_date_order = discharge_date < admission_date

    print(f"\n2.IMPOSSIBLE DATES")
    print("Invalid date format:", invalid_date_format.sum())
    print("Discharge before admission", invalid_date_order.sum())

    if invalid_date_order.sum() > 0:
        print(
            df.loc[
                invalid_date_order,
                ["Name", "Date of Admission", "Discharge Date"]
            ]
        )

    # ------------------------------
    # 3 .Invalid Gender Values
    # -----------------------------
    valid_gender = ["Male", "Female"]

    invalid_gender = ~df["Gender"].isin(valid_gender)

    print("\n3. INVALID GENDER VALUES")
    print("Count:", invalid_gender.sum())

    if invalid_gender.sum() > 0:
        print(df.loc[invalid_gender,["Name","Gender"]])

    print("Unique Gender values:")
    print(df["Gender"].unique())


    ## Medical data check
    print("\n4. MEDICAL DATA CHECK")

    print("Blood Type values:")
    print(df["Blood Type"].value_counts())

    valid_blood_types = [
        "A+", "A-",
        "B+", "B-",
        "AB+", "AB-",
        "O+", "O-"
    ]

    invalid_blood_type = ~df["Blood Type"].isin(valid_blood_types)

    print("Invalid Blood Type:", invalid_blood_type.sum())


    ## # --------------------------------------------------
    # 5. INCONSISTENT TEXT
    # --------------------------------------------------
    print("\n5. INCONSISTENT TEXT")

    text_columns = [
        "Name",
        "Gender",
        "Blood Type",
        "Medical Condition",
        "Doctor",
        "Hospital",
        "Insurance Provider",
        "Admission Type",
        "Medication",
        "Test Results"
    ]

    for column in text_columns:

        # Leading/trailing spaces
        extra_spaces = (
            df[column].astype(str) !=
            df[column].astype(str).str.strip()
        )

        # Different capitalization
        lowercase_unique = (
            df[column]
            .dropna()
            .astype(str)
            .str.lower()
            .nunique()
        )

        original_unique = df[column].dropna().nunique()

        print(
            f"{column}: "
            f"extra spaces = {extra_spaces.sum()}, "
            f"unique values = {original_unique}, "
            f"case-normalized unique = {lowercase_unique}"
        )


    # --------------------------------------------------
    # 6. INVALID BILLING VALUES
    # --------------------------------------------------
    invalid_billing = df["Billing Amount"] < 0

    print("\n6. INVALID BILLING VALUES")
    print("Negative billing amounts:", invalid_billing.sum())

    if invalid_billing.sum() > 0:
        print(
            df.loc[
                invalid_billing,
                ["Name", "Billing Amount"]
            ]
        )


    # --------------------------------------------------
    # FINAL SUMMARY
    # --------------------------------------------------
    print("\n" + "=" * 70)
    print("DATA QUALITY CHECK COMPLETED")
    print("=" * 70)


In [12]:
data_quality_check(df)

************************************************************
DATA QUALITY CHECK
************************************************************

1. INVALID AGE
Count: 0

2.IMPOSSIBLE DATES
Invalid date format: 0
Discharge before admission 0

3. INVALID GENDER VALUES
Count: 0
Unique Gender values:
<ArrowStringArray>
['Male', 'Female']
Length: 2, dtype: str

4. MEDICAL DATA CHECK
Blood Type values:
Blood Type
A-     6969
A+     6956
AB+    6947
AB-    6945
B+     6945
B-     6944
O+     6917
O-     6877
Name: count, dtype: int64
Invalid Blood Type: 0

5. INCONSISTENT TEXT
Name: extra spaces = 0, unique values = 49992, case-normalized unique = 40235
Gender: extra spaces = 0, unique values = 2, case-normalized unique = 2
Blood Type: extra spaces = 0, unique values = 8, case-normalized unique = 8
Medical Condition: extra spaces = 0, unique values = 6, case-normalized unique = 6
Doctor: extra spaces = 0, unique values = 40341, case-normalized unique = 40341
Hospital: extra spaces = 0, unique va

In [13]:
df.loc[df["Billing Amount"] < 0, "Billing Amount"] = pd.NA

In [14]:
print("Negative billing values:", (df["Billing Amount"] < 0).sum())
print("Missing billing values:", df["Billing Amount"].isna().sum())

Negative billing values: 0
Missing billing values: 108


In [16]:
## Now Cleaning the datasset.

df["Name"] = df["Name"].str.strip().str.title()
df["Billing Amount"] = df["Billing Amount"].round(2)

In [17]:
print(df.head(10).to_string())

                 Name  Age  Gender Blood Type Medical Condition Date of Admission            Doctor                     Hospital Insurance Provider  Billing Amount  Room Number Admission Type Discharge Date   Medication  Test Results
0       Bobby Jackson   30    Male         B-            Cancer        2024-01-31     Matthew Smith              Sons and Miller         Blue Cross        18856.28          328         Urgent     2024-02-02  Paracetamol        Normal
1        Leslie Terry   62    Male         A+           Obesity        2019-08-20   Samantha Davies                      Kim Inc           Medicare        33643.33          265      Emergency     2019-08-26    Ibuprofen  Inconclusive
2         Danny Smith   76  Female         A-           Obesity        2022-09-22  Tiffany Mitchell                     Cook PLC              Aetna        27955.10          205      Emergency     2022-10-07      Aspirin        Normal
3        Andrew Watts   28  Female         O+          Diabetes 

In [18]:
df.to_csv("healthcare_cleaned_dataset.csv",index=False)

In [ ]:
class MasterCard:
    def __init__(self,name,payment,money):
        self.name = name
        self.payment = payment
        self.money = money

    def collection(self):
        print("The name of the customer is:",self.name)

    def payment_detail(self):
        print("The payment done is :",self.payment)

In [8]:
print(df.head(20).to_string())

                   Name  Age  Gender Blood Type Medical Condition Date of Admission            Doctor                     Hospital Insurance Provider  Billing Amount  Room Number Admission Type Discharge Date   Medication  Test Results
0         Bobby JacksOn   30    Male         B-            Cancer        2024-01-31     Matthew Smith              Sons and Miller         Blue Cross    18856.281306          328         Urgent     2024-02-02  Paracetamol        Normal
1          LesLie TErRy   62    Male         A+           Obesity        2019-08-20   Samantha Davies                      Kim Inc           Medicare    33643.327287          265      Emergency     2019-08-26    Ibuprofen  Inconclusive
2           DaNnY sMitH   76  Female         A-           Obesity        2022-09-22  Tiffany Mitchell                     Cook PLC              Aetna    27955.096079          205      Emergency     2022-10-07      Aspirin        Normal
3          andrEw waTtS   28  Female         O+         